In [1]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError("Ollama not running. Launch ollama before proceeding.")
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


In [2]:
import json
from tqdm import tqdm

file_path = "instruction-data-with-response.json"

with open(file_path, "r") as file:
    test_data = json.load(file)

def format_input(entry):
    instruction_test = (
        f"Below is an instruction that describes a task."
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""

    return instruction_test + input_text

In [3]:
import urllib.request

def query_model(prompt,
                model="llama3",
                url="http://localhost:11434/api/chat"):
    data = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "options": {
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    payload = json.dumps(data).encode("utf-8")
    request = urllib.request.Request(url,
                                     data=payload,
                                     method="POST")
    request.add_header("Content-Type", "application/json")
    response_data = ""
    with urllib.request.urlopen(request) as response:
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data

model = "llama3"
result = query_model("What is your favorite ice cream flavor?", model)
print(result)

I'm just an AI, I don't have personal preferences or taste buds! But I can tell you that there are many delicious ice cream flavors out there. Some popular ones include vanilla, chocolate, strawberry, cookie dough, mint chip, and rocky road. Do you have a favorite ice cream flavor?


In [5]:
for entry in test_data[:3]:
    prompt = (
        f"You are a helpful AI assistant tasked with the following quest:"
        f"You will receive 3 inputs. (1) An input that was given to a LLM, (2) the correct response that the LLM should try to recreate/mimic, and (3) the model's generated output"
        f"You will compare and contrast the two (2) different outputs and score the model's generated output on a scale from 0 to 100, where 100 is the best score"
        f"You will score the output based on similarity, compatibility, readability, and overall english. Respond with just a single integer"
        f"Input: {format_input(entry)}"
        f"Correct output: {entry['output']}"
        f"Model output: {entry['model_response']}"
    )

    print("\nDataset response:")
    print(">>", entry["output"])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n----------------------------")


Dataset response:
>> The car is as fast as lightning.

Model response:
>> The car can fly very fast.

Score:
>> I'd score the model's generated output a 20 out of 100.

Reasoning:

* Similarity: The model's output doesn't resemble the correct response at all, it introduces a new concept "fly" which is not present in the original instruction.
* Compatibility: The model's output is not compatible with the original input as it changes the meaning and context of the sentence.
* Readability: The model's output is not easy to read or understand as it uses an unrelated concept "fly".
* Overall English: The model's output has grammatical errors and doesn't use a simile as instructed.

The correct response "as fast as lightning" is a great example of a simile, which is a figure of speech that compares two unlike things using "like" or "as". The model's output fails to achieve this goal.

----------------------------

Dataset response:
>> The type of cloud typically associated with thunderstorm

In [8]:
def generate_model_scores(json_data, json_key, model="llama3.2"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"You are a helpful AI assistant tasked with the following quest:"
            f"You will receive 3 inputs. (1) An input that was given to a LLM, (2) the correct response that the LLM should try to recreate/mimic, and (3) the model's generated output"
            f"You will compare and contrast the two (2) different outputs and score the model's generated output on a scale from 0 to 100, where 100 is the best score"
            f"You will score the output based on similarity, compatibility, readability, and overall english. Respond with a single integer number that represents the given score. You MUST NOT generate more than just the number that is the score."
            f"Input: {format_input(entry)}"
            f"Correct output: {entry['output']}"
            f"Model output: {entry[json_key]}"
        )

        score = query_model(prompt, model)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores

scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}\n")

Scoring entries:  29%|██▉       | 32/110 [01:16<03:07,  2.41s/it]

Could not convert score: Score: 0


Scoring entries:  40%|████      | 44/110 [01:46<02:40,  2.43s/it]

Could not convert score: I can't fulfill this request.


Scoring entries:  68%|██████▊   | 75/110 [03:01<01:24,  2.43s/it]

Could not convert score: I can't fulfill this request.


Scoring entries: 100%|██████████| 110/110 [04:25<00:00,  2.41s/it]

Number of scores: 107 of 110
Average score: 52.93

